# Notebook 5: Preprocessing

**Automotive Car Price Prediction Pipeline**

---

This notebook prepares the engineered data for model training:
1. Stratified train/test split (using `price_category` to ensure equal class distribution)
2. Imputation (`volume_cm3` → 0 for EVs; other numeric → train median; categorical → train mode)
3. Scaling (StandardScaler — fit on train only)
4. Encoding (OneHotEncoder — fit on train only)

**Input:** `workspace.default.carprice_project_engineered_data` (from Notebook 4)

**Output:** `workspace.default.carprice_project_preprocessed_data`

## 1. Load Data from Catalog

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import pickle
import os

# Read from catalog and convert to pandas
df_spark = spark.table("workspace.default.carprice_project_engineered_data")
df = df_spark.toPandas()

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (56133, 13)
Columns: ['priceUSD', 'condition', 'mileage_kilometers', 'fuel_type', 'volume_cm3', 'color', 'transmission', 'drive_unit', 'segment', 'car_age', 'mileage_per_year', 'make_category', 'price_category']


,priceUSD,condition,mileage_kilometers,fuel_type,volume_cm3,color,transmission,drive_unit,segment,car_age,mileage_per_year,make_category,price_category
0,16800,with mileage,44000.0,petrol,2000.0,white,auto,front-wheel drive,j,3,14666.67,nissan,luxury
1,16900,with mileage,80000.0,petrol,1200.0,red,auto,front-wheel drive,j,3,26666.67,nissan,luxury
2,9650,with mileage,198000.0,petrol,1600.0,brown,mechanics,front-wheel drive,j,9,22000.00,nissan,luxury
3,9800,with mileage,117000.0,petrol,1600.0,black,mechanics,front-wheel drive,j,8,14625.00,nissan,luxury
4,11500,with mileage,93000.0,petrol,2000.0,other,auto,part-time four-wheel drive,j,8,11625.00,nissan,luxury


In [0]:
print("Data types:")
print(df.dtypes)
print(f"\nNull values coming in:")
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) == 0:
    print("  No null values found.")
else:
    for col_name, cnt in null_counts.items():
        pct = cnt / len(df) * 100
        dtype = df[col_name].dtype
        if col_name == 'volume_cm3':
            strategy = "0 (electric cars — no combustion engine)"
        elif np.issubdtype(dtype, np.number):
            strategy = "median"
        else:
            strategy = "mode"
        print(f"  {col_name:25s}: {cnt} ({pct:.1f}%) -> will impute with {strategy}")

Data types:
priceUSD                int64
condition              object
mileage_kilometers    float64
fuel_type              object
volume_cm3            float64
color                  object
transmission           object
drive_unit             object
segment                object
car_age                 int64
mileage_per_year      float64
make_category          object
price_category         object
dtype: object

Null values coming in:
  volume_cm3               : 47 (0.1%) -> will impute with 0 (electric cars — no combustion engine)
  drive_unit               : 1904 (3.4%) -> will impute with mode
  segment                  : 5281 (9.4%) -> will impute with mode


## 2. Separate Features and Target

In [0]:
target_col = "priceUSD"

# Keep price_category aside for stratification, then drop it from features
stratify_col = df['price_category']

X = df.drop(columns=[target_col, 'price_category'])
y = df[target_col]

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nTarget stats:")
print(f"  Mean:   ${y.mean():,.0f}")
print(f"  Median: ${y.median():,.0f}")
print(f"  Std:    ${y.std():,.0f}")
print(f"  Min:    ${y.min():,.0f}")
print(f"  Max:    ${y.max():,.0f}")
print(f"\nPrice category distribution:")
print(stratify_col.value_counts().to_string())

Features (X): (56133, 11)
Target (y): (56133,)

Target stats:
  Mean:   $7,420
  Median: $5,350
  Std:    $8,322
  Min:    $100
  Max:    $235,235

Price category distribution:
luxury    18905
mid       18659
budget    18569


## 3. Identify Column Types

In [0]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}):")
for col in numeric_cols:
    print(f"  {col}")

print(f"\nCategorical columns ({len(categorical_cols)}):")
for col in categorical_cols:
    print(f"  {col} — {X[col].nunique()} unique values")

Numeric columns (4):
  mileage_kilometers
  volume_cm3
  car_age
  mileage_per_year

Categorical columns (7):
  condition — 3 unique values
  fuel_type — 3 unique values
  color — 13 unique values
  transmission — 2 unique values
  drive_unit — 4 unique values
  segment — 9 unique values
  make_category — 39 unique values


## 4. Stratified Train/Test Split

We use `price_category` (budget / mid / luxury) as the stratification key to ensure both train and test sets have the same class distribution across price ranges.

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_col
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:     {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

# Verify class distribution is similar in both sets
train_dist = stratify_col[X_train.index].value_counts(normalize=True).sort_index()
test_dist  = stratify_col[X_test.index].value_counts(normalize=True).sort_index()

print(f"\nPrice category distribution (stratification check):")
print(f"{'Category':<10} {'Train %':>10} {'Test %':>10}")
print("-" * 32)
for cat in train_dist.index:
    print(f"{cat:<10} {train_dist[cat]*100:>9.1f}% {test_dist[cat]*100:>9.1f}%")

Training set: 44906 samples (80%)
Test set:     11227 samples (20%)

Price category distribution (stratification check):
Category      Train %     Test %
--------------------------------
budget          33.1%      33.1%
luxury          33.7%      33.7%
mid             33.2%      33.2%


## 5. Imputation (fit on train only)

- `volume_cm3` → impute with **0** (missing values are electric cars — no combustion engine, so displacement is genuinely zero, not unknown)
- All other numeric columns → impute with **train median**
- Categorical columns → impute with **train mode** (most frequent value)

In [0]:
# Reset index after train/test split to avoid pandas index alignment issues
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

# volume_cm3: fill missing with 0 (electric cars — no combustion engine)
if 'volume_cm3' in X_train.columns:
    X_train['volume_cm3'] = X_train['volume_cm3'].fillna(0)
    X_test['volume_cm3']  = X_test['volume_cm3'].fillna(0)
    print(f"volume_cm3 nulls filled with 0 (electric cars).")

# Impute numeric columns with train median
num_imputer = SimpleImputer(strategy="median")
X_train[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = num_imputer.transform(X_test[numeric_cols])

# Impute categorical columns — use fillna(mode) directly per column
# (avoids SimpleImputer index alignment issues with Spark-loaded DataFrames)
for col in categorical_cols:
    train_mode = X_train[col].mode()[0]   # most frequent value in training set
    X_train[col] = X_train[col].fillna(train_mode)
    X_test[col]  = X_test[col].fillna(train_mode)  # use TRAIN mode on test too (no leakage)

print(f"\nNulls in X_train after imputation: {X_train.isnull().sum().sum()}")
print(f"Nulls in X_test after imputation:  {X_test.isnull().sum().sum()}")
print("Imputation complete.")

volume_cm3 nulls filled with 0 (electric cars).

Nulls in X_train after imputation: 0
Nulls in X_test after imputation:  0
Imputation complete.


## 6. Scaling and Encoding (fit on train only)

In [0]:
# StandardScaler for numeric columns — fit on train only
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[numeric_cols])
X_test_num  = scaler.transform(X_test[numeric_cols])

# OneHotEncoder for categorical columns — fit on train only
# drop='first' removes one category per group to avoid perfect multicollinearity
# (the "dummy variable trap" — without drop='first', LR coefficients explode to quadrillions)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop='first')
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat  = encoder.transform(X_test[categorical_cols])

# Get feature names
cat_feature_names = encoder.get_feature_names_out(categorical_cols).tolist()
all_feature_names = numeric_cols + cat_feature_names

# Combine numeric and categorical
X_train_processed = np.hstack([X_train_num, X_train_cat])
X_test_processed  = np.hstack([X_test_num,  X_test_cat])

print(f"Preprocessed training shape: {X_train_processed.shape}")
print(f"Preprocessed test shape:     {X_test_processed.shape}")
print(f"\nTotal features: {len(all_feature_names)}")
print(f"  Numeric:  {len(numeric_cols)}")
print(f"  Encoded:  {len(cat_feature_names)}  (one category dropped per group — dummy variable trap fix)")
print(f"\nNulls in X_train after processing: {np.isnan(X_train_processed).sum()}")
print(f"Nulls in X_test after processing:  {np.isnan(X_test_processed).sum()}")

Preprocessed training shape: (44906, 70)
Preprocessed test shape:     (11227, 70)

Total features: 70
  Numeric:  4
  Encoded:  66  (one category dropped per group — dummy variable trap fix)

Nulls in X_train after processing: 0
Nulls in X_test after processing:  0


## 7. Combine and Save to Catalog

In [0]:
import re

def clean_col_name(name):
    """Replace any character invalid for Delta column names with underscore, then collapse multiple underscores."""
    name = re.sub(r"[ ,;{}\(\)\n\t=\-/]", "_", name)
    name = re.sub(r"_+", "_", name)          # collapse multiple underscores
    name = name.strip("_").lower()            # strip leading/trailing, lowercase
    return name

# Clean feature names
all_feature_names_clean = [clean_col_name(c) for c in all_feature_names]

# Verify no duplicates after cleaning
assert len(all_feature_names_clean) == len(set(all_feature_names_clean)), \
    "Duplicate column names after cleaning! Check categories."

print(f"Sample cleaned column names:")
for orig, clean in zip(all_feature_names[:5], all_feature_names_clean[:5]):
    print(f"  '{orig}' → '{clean}'")
print(f"  ... ({len(all_feature_names_clean)} total)")

# Create DataFrames with cleaned column names
train_df = pd.DataFrame(X_train_processed, columns=all_feature_names_clean)
train_df["price"] = y_train.values
train_df["split"] = "train"

test_df = pd.DataFrame(X_test_processed, columns=all_feature_names_clean)
test_df["price"] = y_test.values
test_df["split"] = "test"

# Combine
combined_df = pd.concat([train_df, test_df], ignore_index=True)

print(f"\nCombined shape: {combined_df.shape}")
print(f"\nSplit distribution:")
print(combined_df['split'].value_counts().to_string())

Sample cleaned column names:
  'mileage_kilometers' → 'mileage_kilometers'
  'volume_cm3' → 'volume_cm3'
  'car_age' → 'car_age'
  'mileage_per_year' → 'mileage_per_year'
  'condition_with damage' → 'condition_with_damage'
  ... (70 total)

Combined shape: (56133, 72)

Split distribution:
train    44906
test     11227


In [0]:
# Save to Unity Catalog
combined_spark = spark.createDataFrame(combined_df)
combined_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.carprice_project_preprocessed_data")

# Verify
verify = spark.table("workspace.default.carprice_project_preprocessed_data")
print(f"Saved carprice_project_preprocessed_data: ({verify.count()} rows, {len(verify.columns)} columns)")
print(f"\nSplit distribution:")
verify.groupBy("split").count().show()

Saved carprice_project_preprocessed_data: (56133 rows, 72 columns)

Split distribution:
+-----+-----+
|split|count|
+-----+-----+
| test|11227|
|train|44906|
+-----+-----+



### Preprocessing Summary

| Step | Details |
|------|---------|
| **Stratified Train/Test Split** | 80/20, random_state=42, stratified on `price_category` |
| **`volume_cm3` Imputation** | `fillna(0)` — missing values are electric cars (no combustion engine → displacement is genuinely 0, not unknown) |
| **Other Numeric Imputation** | SimpleImputer(strategy="median") — fit on train only |
| **Categorical Imputation** | SimpleImputer(strategy="most_frequent") — fit on train only |
| **Numeric Scaling** | StandardScaler — fit on train only |
| **Categorical Encoding** | OneHotEncoder(drop='first', handle_unknown='ignore') — fit on train only |
| **Saved Table** | `workspace.default.carprice_project_preprocessed_data` |

> All steps are fit **on training data only** to prevent data leakage into the test set.

> **Why `drop='first'`?** Without it, all OHE columns for a group (e.g. all `transmission_*`) sum to exactly 1 for every row — perfect multicollinearity. Linear Regression has no unique solution and assigns arbitrary coefficients of quadrillions of dollars that cancel each other out. Dropping the first category of each group breaks this perfect collinearity and gives LR meaningful, interpretable coefficients.

> **Why 0 for `volume_cm3` missing values?** The 47 missing entries are electric vehicles. EVs have no combustion engine, so their engine displacement is genuinely zero — not a data entry error. Imputing with the median (~1,600 cm³) would incorrectly signal to models that these EVs have a large petrol engine, corrupting the `volume_cm3` feature for exactly the cars where `fuel_type_electrocar` is already providing a strong price signal.